<a href="https://colab.research.google.com/github/KP-365/Fake_news/blob/main/scaffold_FakeNews_finn's_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Three-Layer Framework for Fake News Detection, Verification, and Explanation

**ECS7036P Group 10 — Notebook Scaffold**

This notebook is a guide, not an implementation. Every section below describes what
needs to happen and which GitHub issue it corresponds to, but contains no code —
fill in the empty cells as you implement each piece. Delete this line and the guide
text once a section is implemented, or move the guide text into a comment above your code.

Run top to bottom: **Setup → Data → Layer 1 (Classifier) → Layer 2 (Verification)
→ Layer 3 (Explanation) → End-to-end demo**.

## 0. Setup
*Owner: Kayleb — [Issue #1](https://github.com/KP-365/Fake_news/issues/1)*

- Confirm you're on a Colab **T4 GPU** runtime (`Runtime → Change runtime type`).
- Install dependencies from `requirements.txt`: `transformers`, `peft`, `torch`,
  `datasets`, `scikit-learn`, `gradio`, `anthropic`, `numpy`, `matplotlib`.
- Set any API keys you'll need later (Google Fact Check Tools API key, Hugging Face
  token) as Colab secrets rather than hardcoding them.

In [1]:
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification
from peft import (
    LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict,
    set_peft_model_state_dict,
)
import copy
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import torch
import re
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

## 1. Data
*Owner: Eric — [Issue #2](https://github.com/KP-365/Fake_news/issues/2), [Issue #9](https://github.com/KP-365/Fake_news/issues/9)*

- Load the **LIAR** dataset (primary, 12,836 labelled political statements from
  PolitiFact) and the **WELFake** dataset (backup, 72,134 labelled articles).
- Build the preprocessing/tokenisation pipeline shared by both datasets so they can
  feed the same classifier interface.
- Produce a train/validation/held-out test split — the test set is what every layer's
  evaluation below will be measured against.

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import os # Import os for path joining

# Define column names for the LIAR dataset (standard format)
liar_column_names = [
    "id", "label", "statement", "subject", "speaker", "speaker_title",
    "state_info", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "context"
]

# Download latest version for fake-news-classification (WELFake)
path_welfake = kagglehub.dataset_download("saurabhshahane/fake-news-classification")
print("Path to WELFake dataset files:", path_welfake)

# Download latest version for liar-dataset
path_liar = kagglehub.dataset_download("doanquanvietnamca/liar-dataset")
print("Path to LIAR dataset files:", path_liar)

# Load the LIAR data sets with explicit column names and no header using pandas directly
train_liar_df = pd.read_csv(os.path.join(path_liar, "train.tsv"), sep='\t', header=None, names=liar_column_names)
valid_liar_df = pd.read_csv(os.path.join(path_liar, "valid.tsv"), sep='\t', header=None, names=liar_column_names)
test_liar_df = pd.read_csv(os.path.join(path_liar, "test.tsv"), sep='\t', header=None, names=liar_column_names)

Using Colab cache for faster access to the 'fake-news-classification' dataset.
Path to WELFake dataset files: /kaggle/input/fake-news-classification
Using Colab cache for faster access to the 'liar-dataset' dataset.
Path to LIAR dataset files: /kaggle/input/liar-dataset


In [3]:

WELFake_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "saurabhshahane/fake-news-classification",
  "WELFake_Dataset.csv",
)



Using Colab cache for faster access to the 'fake-news-classification' dataset.


In [4]:
print("Shape of train_liar_df:", train_liar_df.shape)
print("Shape of valid_liar_df:", valid_liar_df.shape)
print("Shape of test_liar_df:", test_liar_df.shape)
print("Shape of WELFake_df:", WELFake_df.shape)

Shape of train_liar_df: (10240, 14)
Shape of valid_liar_df: (1284, 14)
Shape of test_liar_df: (1267, 14)
Shape of WELFake_df: (72134, 4)


In [5]:
print("\n--- train_liar_df head ---")
print(train_liar_df.head())
print("\n--- valid_liar_df head ---")
print(valid_liar_df.head())
print("\n--- test_liar_df head ---")
print(test_liar_df.head())
print("\n--- WELFake_df head ---")
print(WELFake_df.head())


--- train_liar_df head ---
           id        label                                          statement  \
0   2635.json        false  Says the Annies List political group supports ...   
1  10540.json    half-true  When did the decline of coal start? It started...   
2    324.json  mostly-true  Hillary Clinton agrees with John McCain "by vo...   
3   1123.json        false  Health care reform legislation is likely to ma...   
4   9028.json    half-true  The economic turnaround started at the end of ...   

                              subject         speaker         speaker_title  \
0                            abortion    dwayne-bohac  State representative   
1  energy,history,job-accomplishments  scott-surovell        State delegate   
2                      foreign-policy    barack-obama             President   
3                         health-care    blog-posting                   NaN   
4                        economy,jobs   charlie-crist                   NaN   

  state_in

# Clean and explore data 0 = fake and 1 = real

In [6]:
WELFake_df

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
...,...,...,...,...
72129,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0
72130,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1
72131,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0
72132,72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0


In [7]:
df = WELFake_df.copy()

print("Initial shape:", df.shape)
print("Columns:", list(df.columns))

# drop the stray index column WELFake ships with
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("\nMissing values:\n", df.isna().sum())
print("\nLabel distribution:\n", df["label"].value_counts())
# IMPORTANT: confirm which label value means real vs fake before training —
# print a few rows of each label and read them to check the direction.
print("\nSample label=0:\n", df[df["label"] == 0][["title", "text"]].head(2))
print("\nSample label=1:\n", df[df["label"] == 1][["title", "text"]].head(2))

# ---- 2. Handle missing values ----
# missing title -> fill with empty string (text matters more than title)
# missing text -> drop the row (no content to classify)
df["title"] = df["title"].fillna("")
df = df.dropna(subset=["text"])


# ---- 3. Clean text ----
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)   # URLs
    text = re.sub(r"<.*?>", "", text)              # HTML tags
    text = re.sub(r"\s+", " ", text).strip()       # collapse whitespace
    return text


df["title"] = df["title"].apply(clean_text)
df["text"] = df["text"].apply(clean_text)

# drop rows that are now empty/whitespace-only after cleaning
df = df[df["text"].str.len() > 0]

# combine title + text — headline sensationalism/clickbait is itself a
# known fake-news signal, so keep it rather than discarding it
df["content"] = (df["title"] + ". " + df["text"]).str.strip(". ")

# ---- 4. Remove duplicates ----
before = len(df)
df = df.drop_duplicates(subset=["text"])
print(f"\nDropped {before - len(df)} duplicate rows")

# ---- 5. Split (WELFake ships as one CSV) ----
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=42
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print("\nFinal shape:", df.shape)
print("Train:", train_df.shape, "Valid:", valid_df.shape, "Test:", test_df.shape)

train_df.to_csv("welfake_train.csv", index=False)
valid_df.to_csv("welfake_valid.csv", index=False)
test_df.to_csv("welfake_test.csv", index=False)
print("\nSaved welfake_train.csv, welfake_valid.csv, welfake_test.csv")

Initial shape: (72134, 4)
Columns: ['Unnamed: 0', 'title', 'text', 'label']

Missing values:
 title    558
text      39
label      0
dtype: int64

Label distribution:
 label
1    37106
0    35028
Name: count, dtype: int64

Sample label=0:
                                                 title  \
3   Bobby Jindal, raised Hindu, uses story of Chri...   
11  May Brexit offer would hurt, cost EU citizens ...   

                                                 text  
3   A dozen politically active pastors came here f...  
11  BRUSSELS (Reuters) - British Prime Minister Th...  

Sample label=1:
                                                title  \
0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1                                                NaN   

                                                text  
0  No comment is expected from Barack Obama Membe...  
1     Did they post their votes for Hillary already?  

Dropped 8618 duplicate rows

Final shape: (62649, 4)
Train: (43854,

In [8]:
# Create a df
welfake_train_df = pd.read_csv("welfake_train.csv")
welfake_train_df = welfake_train_df[["title", "text", "label"]]

welfake_valid_df = pd.read_csv("welfake_valid.csv")
welfake_valid_df = welfake_valid_df[["title", "text", "label"]]

welfake_test_df = pd.read_csv("welfake_test.csv")
welfake_test_df = welfake_test_df[["title", "text", "label"]]

In [9]:
welfake_train_df

,title,text,label
0,"Dana Perino: Conservatives, here are 5 reasons...","Wednesday, I participated in Facebook’s meetin...",0
1,Spain passes measures to control Catalan finan...,MADRID (Reuters) - The Spanish government on F...,0
2,Trump says U.S. should spend less on NATO,WASHINGTON (Reuters) - Republican presidential...,0
3,The Billion-Dollar Jackpot: Engineered to Drai...,If you’ve noticed that colossal lottery winnin...,0
4,NaN,Well we know that they think we are a basket f...,1
...,...,...,...
43849,[VIDEO] MSNBC ANALYST AT LOSS TO EXPLAIN: Trum...,Trump is not your traditional Republican candi...,1
43850,"U.S., EU set meeting on airline security, elec...",WASHINGTON/BRUSSELS (Reuters) - U.S. and Europ...,0
43851,"President Obama BLASTS ‘Insecure’ Trump, Makes...",Donald Trump fell on his face repeatedly durin...,1
43852,Eli Lilly’s Experimental Alzheimer’s Drug Fail...,An experimental Alzheimer’s drug that had prev...,0


In [10]:
reuters_rate = df["text"].str.contains(r"\(Reuters\)", regex=True).groupby(df["label"]).mean()
print(reuters_rate)

label
0    0.607805
1    0.000464
Name: text, dtype: float64


## 2. Layer 1 — Classification (BERT-LoRA)
*Owner: Kayleb — [Issue #3](https://github.com/KP-365/Fake_news/issues/3)*

- Load `robert-base-uncased` from Hugging Face Transformers.
- Attach a LoRA adapter via Hugging Face **PEFT** and fine-tune on the LIAR training
  split to classify a claim as real or fake.
- Keep the base weights frozen — only the LoRA low-rank updates should be trainable.

In [11]:
#Uninstalling torchao so Lora can be attached
!pip install -q -U torchao

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    id2label={0: "fake", 1: "real"},
    label2id={"fake": 0, "real": 1},
)
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 294,912 || all params: 124,942,082 || trainable%: 0.2360


Going from 109,809,210 trainable parameters to 294,912 trainable parameters

# Training the model on the desired dataset

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)


class WelfakeDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.titles = df["title"].fillna("").tolist()
        self.texts = df["text"].fillna("").tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        # tells DataLoader how many examples exist
        return len(self.labels)

    def __getitem__(self, idx):
        # called once per example, whenever DataLoader needs it
        content = (self.titles[idx] + ". " + self.texts[idx]).strip(". ")

        encoding = self.tokenizer(
            content,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

Using device: cuda


Train model

In [14]:
train_dataset = WelfakeDataset(train_df, tokenizer)
valid_dataset = WelfakeDataset(valid_df, tokenizer)
test_dataset = WelfakeDataset(test_df, tokenizer)

#Data loader to batch the samples together and shuffle them each epoch

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Combine the model + Lora

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "fake", 1: "real"},
    label2id={"fake": 0, "real": 1},
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.to(device)

# optimiser

optimiser = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)

num_epochs = 3
early_stopping_patience = 1
best_valid_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None

# Evaluation loops
def evaluate(model, dataloader, max_batches=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    batches_evaluated = 0

    with torch.no_grad():
        for batch_index, batch in enumerate(dataloader):
            if max_batches is not None and batch_index >= max_batches:
                break

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            preds = torch.argmax(outputs.logits, dim=1)

            total_loss += outputs.loss.item()
            batches_evaluated += 1
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    valid_loss = total_loss / batches_evaluated
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return acc, f1, valid_loss

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        optimiser.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimiser.step()

        if step % 50 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Step {step}/{len(train_loader)}, Loss: {loss.item():.4f}")

        if step % 300 == 0 and step > 0:
            valid_acc, valid_f1, valid_loss = evaluate(model, valid_loader, max_batches=50)
            model.train()
            print(f"  [Step {step}] Val_loss: {valid_loss:.4f}")

    avg_train_loss = total_loss / len(train_loader)
    valid_acc, valid_f1, valid_loss = evaluate(model, valid_loader)

    print(f"Epoch {epoch+1}/{num_epochs}, Avg Train Loss: {avg_train_loss:.4f}, Valid F1: {valid_f1:.4f}, Valid Loss: {valid_loss:.4f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_model_state = copy.deepcopy(get_peft_model_state_dict(model))
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= early_stopping_patience:
            print("Early stopping triggered")
            break

if best_model_state is None:
    raise RuntimeError("Training completed without a best model state")
set_peft_model_state_dict(model, best_model_state)

test_acc, test_f1, _ = evaluate(model, test_loader)
print(f"Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}")

checkpoint_dir = "models/roberta-trained-welfake"
os.makedirs(checkpoint_dir, exist_ok=True)
model.save_pretrained(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)
print(f"Saved best model and tokenizer to {checkpoint_dir}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 294,912 || all params: 124,942,082 || trainable%: 0.2360
Epoch 1/3, Step 0/1371, Loss: 0.7248
Epoch 1/3, Step 50/1371, Loss: 0.6599
Epoch 1/3, Step 100/1371, Loss: 0.1305
Epoch 1/3, Step 150/1371, Loss: 0.0543
Epoch 1/3, Step 200/1371, Loss: 0.0544
Epoch 1/3, Step 250/1371, Loss: 0.1033
Epoch 1/3, Step 300/1371, Loss: 0.0102
  [Step 300] Val_loss: 0.1319
Epoch 1/3, Step 350/1371, Loss: 0.0222
Epoch 1/3, Step 400/1371, Loss: 0.0139
Epoch 1/3, Step 450/1371, Loss: 0.0778
Epoch 1/3, Step 500/1371, Loss: 0.0088
Epoch 1/3, Step 550/1371, Loss: 0.0123
Epoch 1/3, Step 600/1371, Loss: 0.1836
  [Step 600] Val_loss: 0.0501
Epoch 1/3, Step 650/1371, Loss: 0.0063
Epoch 1/3, Step 700/1371, Loss: 0.0149
Epoch 1/3, Step 750/1371, Loss: 0.0440
Epoch 1/3, Step 800/1371, Loss: 0.0120
Epoch 1/3, Step 850/1371, Loss: 0.0553
Epoch 1/3, Step 900/1371, Loss: 0.0048
  [Step 900] Val_loss: 0.0194
Epoch 1/3, Step 950/1371, Loss: 0.0065
Epoch 1/3, Step 1000/1371, Loss: 0.0055
Epoch 1/3, Step 10

## 3. Layer 1 — Monte Carlo Dropout (uncertainty)
*Owner: Kayleb — [Issue #4](https://github.com/KP-365/Fake_news/issues/4)*

- Keep dropout active at inference time and run multiple stochastic forward passes
  per input.
- Use the spread across passes to produce a per-prediction uncertainty estimate,
  so the model can say "uncertain" instead of forcing a binary label.

In [14]:
torch.save(model.base_model.model.classifier.state_dict(), f"{checkpoint_dir}/classifier_head.pt")

## 4. Layer 1 — Evaluation: F1, confusion matrix, calibration
*Owner: Kayleb — [Issue #5](https://github.com/KP-365/Fake_news/issues/5), [Issue #6](https://github.com/KP-365/Fake_news/issues/6), [Issue #7](https://github.com/KP-365/Fake_news/issues/7)*

- Compute per-class precision/recall, macro-F1, and a 2x2 confusion matrix on the
  held-out LIAR test set. Target: macro-F1 competitive with published LIAR baselines.
- Plot a reliability diagram and compute Expected Calibration Error (ECE); compare
  against an uncalibrated softmax baseline. Target: lower ECE.
- Evaluate accuracy-on-retained predictions as the least-confident cases are deferred
  (selective classification) — confirm accuracy rises as coverage shrinks.

## 5. Layer 2 — Fact Check verification
*Owner: Eric — [Issue #10](https://github.com/KP-365/Fake_news/issues/10)*

- Query the **Google Fact Check Tools API** for each test claim, searching verified
  sources (PolitiFact, Snopes, etc.) for supporting or refuting evidence.
- Store the retrieved verdict alongside the classifier's prediction for the next step.

## 6. Layer 2 — Conflict detection & evaluation
*Owner: Eric — [Issue #11](https://github.com/KP-365/Fake_news/issues/11), [Issue #12](https://github.com/KP-365/Fake_news/issues/12)*

- Flag cases where the classifier's prediction disagrees with the retrieved
  fact-check verdict.
- Report the verification layer's coverage of the test claims and analyse the
  conflict cases as the hardest examples.

## 7. Baseline models
*Owner: William — [Issue #14](https://github.com/KP-365/Fake_news/issues/14)*

- Implement one or more baseline classifiers (e.g. TF-IDF + logistic regression, or a
  non-LoRA fine-tuned model) purely for comparison against BERT-LoRA.
- Produce evaluation plots comparing the baseline to Layer 1.

## 8. Layer 3 — Explanation agent
*Owner: William — [Issue #15](https://github.com/KP-365/Fake_news/issues/15)*

- Combine the Layer 1 signal (prediction + uncertainty) and the Layer 2 signal
  (fact-check evidence / conflict flag) into a single natural-language explanation.
- The explanation should say *why* the system reached its verdict, not just restate
  the label.

## 9. Layer 3 — Gradio UI + Hugging Face Spaces
*Owner: William — [Issue #16](https://github.com/KP-365/Fake_news/issues/16)*

- Build a Gradio interface: input a claim, output the prediction, calibrated
  confidence, any conflicting fact-checks, and the generated explanation.
- Deploy to Hugging Face Spaces to get the public demo URL required for the
  project's success criteria.

  [Guide](https://gradio.app/guides/quickstart)

## 10. End-to-end check & faithfulness review
*Owner: Team — [Issue #19](https://github.com/KP-365/Fake_news/issues/19), [Issue #20](https://github.com/KP-365/Fake_news/issues/20)*

- Run a handful of claims through the full pipeline (classify → confidence →
  conflicts → explanation) and sanity-check the output.
- Spot-check a sample of explanations against the underlying signals to confirm
  each justification is faithful, not just plausible-sounding.

---
Full task tracker: [TASKS.md](../TASKS.md) · [GitHub Issues](https://github.com/KP-365/Fake_news/issues) · [Project board](https://github.com/users/KP-365/projects/8)